# RoPE for a ragged decode batch with a cos/sin cache

In a serving engine, the sequences in one decode batch sit at **different** positions: one request is on token 17, another on token 950. Engines precompute a cos/sin table once and gather rows from it per token, instead of recomputing angles every step.

Write `rope_decode(q, k, positions, max_pos, base=10000.0, scale=1.0)` in PyTorch.

## Shapes
- `q`: `(B, H, T, d)` and `k`: `(B, H_kv, T, d)`, float64, $d$ even. $H$ and $H_{kv}$ may differ (grouped-query attention).
- `positions`: int64 `(B, T)`: the position of each new token in its own sequence. All values are in $[0, \text{max\_pos})$.

## Computation
1. Build the tables once: $\theta_i = \text{base}^{-2i/d}$ for $i = 0, \dots, d/2 - 1$, and for every $m \in [0, \text{max\_pos})$ the angle $(m / \text{scale})\,\theta_i$, giving `cos` and `sin` of shape `(max_pos, d/2)`. (`scale > 1` is linear position interpolation, used to stretch a model to longer contexts.)
2. Gather the rows for `positions`, giving `(B, T, d/2)`, and broadcast them over the head axis of both `q` and `k`.
3. Rotate with the **half** layout (Llama, GPT-NeoX): pair $i$ is features $(i, i + d/2)$:
$$(a, b) \mapsto (a\cos\phi - b\sin\phi,\;\; a\sin\phi + b\cos\phi).$$

## Output
A tuple `(q_rot, k_rot)` with the same shapes as `q` and `k`. No Python loops over batch, heads or tokens.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def rope_decode(q, k, positions, max_pos, base=10000.0, scale=1.0):
    """q: (B,H,T,d), k: (B,Hkv,T,d), positions: (B,T) int64. Return (q_rot, k_rot)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[1.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[1]], "dtype": "int64"}, 4], "e": {"__tup__": [{"__nd__": [[[[0.5403023058681398, 0.8414709848078965]]]], "dtype": "float64"}, {"__nd__": [[[[-0.8414709848078965, 0.5403023058681398]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[1.0, 1.0, 1.0, 1.0]]], [[[1.0, 1.0, 1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0, 1.0, 1.0]]], [[[1.0, 1.0, 1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[0], [3]], "dtype": "int64"}, 8, 100.0], "e": {"__tup__": [{"__nd__": [[[[1.0, 1.0, 1.0, 1.0]]], [[[-1.1311125046603125, 0.6598162824642664, -0.8488724885405782, 1.2508566957869456]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0, 1.0, 1.0]]], [[[-1.1311125046603125, 0.6598162824642664, -0.8488724885405782, 1.2508566957869456]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.8019314252534474, -1.324358995628145, -0.24836162209524854, 0.4204452380655215, 1.1360465324896427, 0.10970639932180819, -0.5526473205362324, -0.7847803553442784]], [[0.7487457707345911, 1.6347830429585775, 0.27276877584472176, -1.2333286640307717, -0.9582652054360887, 1.6000190889991115, 0.2028824405086084, -1.7321348424395848]], [[-0.08369619281702581, -1.1632259734447485, -0.6292880940615545, -0.48800582327685743, -0.7133133716322436, 0.5533784703532895, -0.06308597192528916, -0.5894312580326048]], [[0.40963782655711695, 0.8298553070613239, -1.643023371405677, -0.256730126365494, -0.9807473560440125, -0.17315522486203205, -1.2894187467538587, 0.0206903940375912]]], [[[-0.03788574104406823, -0.304337750958489, -1.0479265051202462, -0.3961903304730927, -1.091328901695709, -1.3552087462047395, 0.22478573245989314, -1.109349937891366]], [[1.1702961011782933, 0.7165876558738361, -1.9978166924497212, 0.272128869412488, -1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208]], [[-0.23342252376577002, -0.255790031399391, 0.9620005318430944, -1.1814468079562157, 0.7380418978456841, -1.0989727630364063, -0.33129089269991674, -0.8404731684222111]], [[1.448731288921672, 0.5682130997882933, 2.4317325028452124, 0.6419163790823205, 0.8449927337191754, 0.8406828762653401, -0.6066115359095516, -0.07002844663838208]]], [[[1.350388867744626, -0.3965507651729716, 0.18879953129109867, -0.021223460417289796, 0.6092164928327407, -0.3649087419473726, -0.15236188875684828, 0.24238142867215043]], [[0.10302313848680769, -0.8649727462818201, 0.8957830431894438, -1.298481208246912, -1.2011155488035266, -1.282491794740462, 0.9669722789332148, -0.36060836889927]], [[-0.9710363785210655, -1.1360213941896466, 0.42113113746240616, -1.054840662577835, -1.2720782100976422, 0.6139930624688609, -1.1967077271925706, -0.32243815075139726]], [[-0.006761540346521785, -0.4453353669298123, -0.05409396159544828, 1.3387734990547715, -0.516894152589275, -1.2593071588782276, -1.8367457051936795, -0.20476611683386706]]]], "dtype": "float64"}, {"__nd__": [[[[-0.3522570104112305, 0.2650909069303235, -0.4642445916374816, -0.4786384630527245, -0.7213157478486046, -0.5197597723701788, 0.1602267063174243, -0.3803527272151388]], [[0.1004418528325065, 1.9011872000332568, 0.4791181507969435, -1.5760898454352261, 1.7335295856131068, 0.34781036998853543, -0.9414132864509847, 0.9070489576858102]]], [[[0.01765561090983897, -0.6152185330350691, -0.6335089254336164, -0.9934317902518573, 0.048118983989838576, 1.0688166937495658, -0.3250520988063787, 0.42082412505635486]], [[1.875646053775279, -1.2145906329798497, 0.2575223239234917, -0.30643692685187435, -1.0592560998278446, -1.0258471994365972, -0.0152851431653622, 0.4338634112001693]]], [[[-0.531589820743032, 0.05439054435949247, 0.7222086544347099, 0.2965612469178253, 0.8686701897408925, 0.4613956081509272, 0.4880056756807095, 1.8266340649418156]], [[0.6231718736925668, 0.12947113235560923, 0.7099438034419987, -0.9191236755082103, -0.3377047857608698, 0.793220854194603, 0.6307867000938839, 1.5483616908312823]]]], "dtype": "float64"}, {"__nd__": [[17], [0], [950]], "dtype": "int64"}, 1024], "e": {"__tup__": [{"__nd__": [[[[1.3128544148552233, 0.061844389368553826, -0.151283259528125, 0.43372510863759783, 0.4583765048362706, -1.3274552779122073, -0.5866991832380596, -0.7775197325364603]], [[-1.1273011506890498, -1.7973154210655324, 0.23451262249143487, -1.2037055783185922, -0.45616285337316814, 1.4150031660411504, 0.2461055050882585, -1.7528501323951906]], [[-0.6627475626136463, -0.39889069361146356, -0.6095437265811239, -0.47791545938609153, 0.2767429982693238, -1.2248300336361755, -0.16864101092847575, -0.5976417866730903]], [[-1.0556053600158777, 0.0647896558648878, -1.4011919140269846, -0.2570447495125567, -0.12395906275665663, 0.8452484031619193, -1.5488020285146076, 0.016323202415589603]]], [[[-0.03788574104406823, -0.304337750958489, -1.0479265051202462, -0.3961903304730927, -1.091328901695709, -1.3552087462047395, 0.22478573245989314, -1.109349937891366]], [[1.1702961011782933, 0.7165876558738361, -1.9978166924497212, 0.272128869412488, -1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208]], [[-0.23342252376577002, -0.255790031399391, 0.9620005318430944, -1.1814468079562157, 0.7380418978456841, -1.0989727630364063, -0.33129089269991674, -0.8404731684222111]], [[1.448731288921672, 0.5682130997882933, 2.4317325028452124, 0.6419163790823205, 0.8449927337191754, 0.8406828762653401, -0.6066115359095516, -0.07002844663838208]]], [[[-0.1361382701801353, -0.04022271157640675, -0.19971580236216516, -0.20950214017957042, 1.4751817517378658, -0.5373946713378582, 0.13774253681462864, 0.1237256864899793]], [[1.1691696052947615, 0.24469731249330806, -0.820580858444031, -0.46198012240438757, -0.2938277772712449, -1.5274443625368086, -1.0315569317523356, -1.2659645375735369]], [[0.8864150132425593, -1.2490107394603103, -0.5098741709486478, -0.35130678434360496, -1.3324275009600788, -0.32787842492193386, 1.161675147819827, -1.0455805697132012]], [[0.4865028255361896, 0.5352642580465348, -0.08409250542738383, 0.9453018393107327, -0.17475778710588585, -1.2237933990254877, 1.835616897056535, 0.9698701340748268]]]], "dtype": "float64"}, {"__nd__": [[[[-0.596542935998045, 0.48127197230251195, -0.48465992819610104, -0.4721036165388129, 0.537138655273554, 0.32984950899992505, 0.07937501044366783, -0.38843422952211826]], [[1.6389730802019684, -0.5898688079774937, 0.6314820622200037, -1.5912811955068358, -0.5735683327814114, 1.8405269931237267, -0.8467842602598951, 0.8801256554318169]]], [[[0.01765561090983897, -0.6152185330350691, -0.6335089254336164, -0.9934317902518573, 0.048118983989838576, 1.0688166937495658, -0.3250520988063787, 0.42082412505635486]], [[1.875646053775279, -1.2145906329798497, 0.2575223239234917, -0.30643692685187435, -1.0592560998278446, -1.0258471994365972, -0.0152851431653622, 0.4338634112001693]]], [[[-0.994448807169951, -0.2755394169375005, -0.6834921878472173, -1.3133078076776903, -0.21965246636527547, 0.3740618508354308, -0.5409004614425877, 1.303749662577956]], [[0.5222702113752589, -0.44744104324332823, -0.660531865972053, -1.7941001055130714, 0.4791884105415549, 0.6676515636647468, -0.6823560061262012, 0.15302636345288723]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.010473028918682835, -1.4623277904204515, 1.9472469308762064, 1.0928928105587499, -1.0587374336035666, 1.3758236699684163], [0.03240279877908362, -1.813928769904867, -0.4203539214229468, -0.5089038339937678, 1.590848160322242, -0.7920686954195238], [-0.25361734037936806, -0.2767608068453714, -0.3789727629772628, -0.9140740610326185, 0.21908690902567676, 1.0768046464135546]], [[0.6239776511717521, -0.9274991160593102, -1.1498157571616898, 0.11897355358204866, -0.7065801636237695, -0.6301656488493008], [-1.6796945788401643, 1.9504916026999646, 0.9166191525408893, -0.9739073319069764, 0.9082102667945468, 1.3424871058246228], [-2.389553349676638, -0.548944785564265, -0.3879732181256381, 0.6482544703432906, -0.12146542230081914, -0.2304313358523204]]], [[[-0.05837138907215449, 1.8533257078420022, 2.159980469779012, -0.5248264370136562, -0.9262439932504336, 2.6925531473868567], [-0.9797446331927372, -0.5734020178922011, 0.036581509348738155, 0.4832462441452338, 1.029035549437811, 0.39097722167013954], [-0.8723830716951919, 0.5063823263497103, 0.24991757985122218, 1.876918992454176, -0.01499979815571943, -1.3369728999172332]], [[-1.0450199143839616, 1.450153058953162, -0.5401312070783946, -2.1044660100855603, -0.5806997054894565, 1.5099831293121058e-05], [1.1888306785373703, -1.014468137602427, 0.6666833259020761, 0.7952990996016167, -0.6993883083236738, -0.18758970531896946], [1.7694502363979239, 1.720484746826155, 0.8555220049018919, 0.33194635951240653, 1.1383096209632015, -0.14067728561527593]]]], "dtype": "float64"}, {"__nd__": [[[[-0.09509306569845781, -0.8603601395273159, 0.005563384668739004, -0.0820166249133826, 2.7736316673693673, -0.19284316578043667], [1.2708179900134384, 1.3204972743933365, -0.18647870191497867, 1.1693971923001296, -2.1768136494850046, 0.09492718485511631], [0.8572613994662929, -2.39786527267812, -1.159171198248098, 1.0560736173734917, -0.25701076898467345, -1.0981986938209587]]], [[[-0.3735956685207676, -0.5425883064344673, 0.7244473133352144, 0.4538858794596457, -0.2802283393557666, -0.6653426856286421], [-0.0538756638943387, 1.325155927345223, 0.3490261125780868, 0.6389455936428303, -0.1506815972085555, -1.2918162500762125], [-0.7108435355871252, 0.6512365658972803, 0.10321610258141592, -0.9038398688268016, -1.617870425602848, 0.11068190035142705]]]], "dtype": "float64"}, {"__nd__": [[4, 5, 6], [0, 1, 2]], "dtype": "int64"}, 16], "e": {"__tup__": [{"__nd__": [[[[0.8202583775911395, -1.2417546108894737, 1.935318282553002, -0.722288428328522, -1.310485790187806, 1.3925532398685034], [-0.478808791168964, -2.13119554628184, -0.41179739642479124, -0.17542860405757546, 1.1309911560209889, -0.7965507779275586], [-0.4989222935657057, -0.3263262823210973, -0.3928601448360387, -0.8068021376873656, 0.13456141861104798, 1.0718159861729286]], [[-0.3178195290233299, -0.7811252975636545, -1.144342525930121, -0.5499941477735433, -0.8656520128917202, -0.6400509384063433], [-1.4103692169825441, 1.6893094443706886, 0.9021047466645139, 1.3344392234625178, 1.3324777604425873, 1.3522830052010204], [-2.1132457789347083, -0.494402068675102, -0.38496219133097975, 1.2901129202847086, -0.2676953882731134, -0.23542712199553167]]], [[[-0.05837138907215449, 1.8533257078420022, 2.159980469779012, -0.5248264370136562, -0.9262439932504336, 2.6925531473868567], [-0.9359959774415778, -0.6205309005920978, 0.035739090212858454, -0.563327621339156, 1.0013218451327153, 0.39105512670452286], [-1.3436381546556004, 0.5055924163457679, 0.25567608358965543, -1.5743295834637774, 0.03200567008679057, -1.3358836297043406]], [[-1.0450199143839616, 1.450153058953162, -0.5401312070783946, -2.1044660100855603, -0.5806997054894565, 1.5099831293121058e-05], [-0.02689315965807937, -0.9809244567056816, 0.6670859281237167, 1.4300684592082897, -0.745705584173171, -0.18615294538854646], [-1.0381890888533731, 1.6075571298225073, 0.8561202211241727, 1.470818119438298, 1.2928946715009961, -0.1369896585211173]]]], "dtype": "float64"}, {"__nd__": [[[[8.658939097895557e-05, -1.35758283503064, 0.007225029540974726, 0.12557631308646974, 2.5671425830973646, -0.19278806183300193], [1.4818463627927088, 1.7857659039828064, -0.18749043487535838, -0.8869044560600885, -1.814737130395729, 0.09291293517407513], [1.1182002595930003, -2.2348213000065367, -1.1448787636401767, 0.7744783868988119, -0.9063035685281728, -1.1130906766852966]]], [[[-0.3735956685207676, -0.5425883064344673, 0.7244473133352144, 0.4538858794596457, -0.2802283393557666, -0.6653426856286421], [-0.5667633233535865, 1.3307202117624737, 0.3518084341498312, 0.29988896961516026, -0.0890331035196216, -1.2910612986544763], [1.1176745556012406, 0.7984066501097267, 0.1027382320338858, -0.2702380956275852, -1.5505355848424078, 0.11112561620100042]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.339531008698164, -0.350530218612727, -0.3294499574610804, 0.28390511348310316, 0.29444772495280114, 0.9621285502204014, 0.9284215804413863, 1.3320930804934341, 0.8063630567820633, -0.3356080333276904, -0.05426894624359061, 0.4942150948314312, -0.31029227000142345, 0.5494387802798892, 1.1536094769949883, -0.8012460099660716], [-1.770384050846499, 1.1936549552915572, 0.01775991033501458, 0.7086698231113021, -1.3323053543858547, 1.7329925604781644, 0.7386198894684642, -0.12607417369146828, -0.3519880388050872, 0.11389588045524304, -0.16673134056534972, -0.2786901272438552, -1.1195999755255746, 1.262809438921045, -0.039577684569559124, 0.07463563738960281]], [[0.09291598860047003, -2.358217336741871, 0.5271414581243252, -0.32915493062535894, -0.2892284645923461, 0.06978787257481246, -0.9724756053284209, 0.8433808099877073, -0.6958646238070096, -0.9036580547128793, -0.8621896009823745, 0.15875269513992676, 0.7457794743624686, -0.5950532355462611, -1.510186675915287, 1.062490693153603], [-1.0650835950433684, 1.194830368529916, -0.05429587697711575, -0.6983498940240875, -0.6400399491357172, 0.4618661736925272, 0.8893717949967992, -0.19817022758122477, 0.946934382371629, 2.08697820766597, -0.20830834526001082, -1.1494466412305686, -1.0544003485304514, -1.3903353550011621, -0.639665450940511, -0.7237792369318543]], [[-1.3251033289662522, 0.5301842547135, -2.6113390049731797, -1.3207989716931114, 2.0152518061838767, 1.0378024284634166, 0.5227916833432323, 0.45725905895534386, 1.7709371728004815, 0.5356296362853685, 0.0775770872272132, -1.2301601607845671, -0.07366681732892769, -0.7338839983148214, -0.7061752749030227, -0.7142389333341079], [-0.04464686469247794, -0.32971081974207533, 0.8711105109344268, -1.3261814419117322, -0.16947019674286637, 1.8594974964345012, 0.0388973371949769, 1.198615221448642, 1.1368634972033143, 1.3166478203373606, 0.7381191695252466, -1.3804460312528912, -0.2673877057870027, -0.38033558246862037, -1.5488553043334206, -0.4638816211751379]], [[-1.2500798351337994, -1.6655514460193832, 0.21089625656145172, -0.04359127244566091, -0.6878149095751852, -0.9505863626128569, -0.541273442072544, -0.39409036937385966, -0.4081102825826052, -0.052707747928468524, -0.43554067396098334, 0.3869719823711992, 0.5228708948391112, 0.35786967764803124, -0.6631035145259176, -0.6165061466820784], [0.4348641034709542, -0.197042778476594, -0.04315170860457781, -2.4919952503112044, -2.700103979886236, 0.33714819206874613, 0.3295586563507561, 1.3934318714435818, 0.7211631188958838, 0.22173880016804176, 0.01636245608868217, -1.426740428209398, 0.8831758896325703, -0.6035860153728927, 0.5024844545833684, 0.20004164543806374]], [[0.272097020758701, 0.8926752155435728, -0.48986611197868063, -0.3872351816195507, -0.8558614733519333, -0.37715202345150317, 1.6659407140512403, -0.3633186666203786, 1.1856163121936107, 0.8260114337421408, 0.7292817915893176, 0.8030151370721609, -1.0142765548969686, 0.37344767888543307, 1.588412097484749, -0.6957959363309963], [-0.0471070855877405, -0.6891591571615298, 0.31701151121305016, -1.0785301318658582, 1.6096797942508279, -0.37683478895943645, -1.542661572000243, 1.790790910814277, 0.970522894800899, 0.518511481410699, 0.5325875434742989, -0.3577980078127549, -0.22329559977220623, 0.7240192298296324, 0.9176249577098493, 0.9326124031440995]], [[0.2125509856107767, -0.466388093105908, -0.19208732917753973, 1.1095997115146592, 1.4399181149572342, -0.34131239008610353, -0.47730400254445376, 1.2475496699476993, 0.7236490231334424, 0.6972581900253083, -1.3123336629533346, -0.3737574414663936, -0.5149955292041763, 0.20790465904469757, -1.6849419797309495, 1.2717528081338776], [0.30013179535055773, 0.43483702547979786, -1.1841403768848728, -0.026420371368168806, 1.3346502881568998, -1.5696956024589206, 0.3057680573775562, 0.6988049143143882, -0.6919471030872025, 0.8147412282685731, -0.33006903056415593, 1.2297665530315567, 0.5275117754251525, 0.22436588572838914, 2.008915981414082, 0.8318964583740944]], [[1.3086357086483342, -1.1895196722305454, -0.6466070677186754, 0.8365690029317673, 0.9056118633317605, -0.05574002999013012, 0.7797265342256705, -1.0583537461465884, 0.21383522572037922, -1.1472838649378465, 0.3405705337781045, -0.4861719071399926, -0.6409975166442754, 0.05629903018263689, 1.8174600636946423, -1.011761734455417], [-0.6017506161418752, 1.067664145735261, -0.4235666699986711, 1.7692642772096374, 0.22252970384637974, 0.09063191415105347, 0.35688413886287884, 1.741179596515403, 0.6553615344906699, 0.5167121329088065, 0.2518577631500006, -1.562418833300465, -0.8852720281451512, 1.4833441210372633, 0.8218783319217637, 0.5060648618402573]], [[0.6370627277466291, -0.5435930521122606, 0.33796426767211185, -0.900794630349013, 0.39136232136016047, -0.7141351107069186, -0.21381982915618122, 0.4534148448314499, -0.32678770337045643, 0.6475819228802786, 0.45603827622850734, 0.35945813704442325, 1.3244113828426742, -1.4738846764712352, -1.5962822788262483, 0.6162417919107772], [0.9407940950142847, 2.0116404277145703, 0.7562567989915012, 0.8302149983209312, -0.8830800515074957, 0.7526217774690592, 0.6738389380752741, -1.0558651891629185, -0.896317509048086, -0.11103120279927593, 0.2882537511901322, -2.944785538646996, 0.0703055101868928, 0.23085449610631834, -0.17683635976985834, 0.26682489806649157]]]], "dtype": "float64"}, {"__nd__": [[[[0.4559062316030521, -1.011638037208477, 0.4161801684774495, -0.25130719973217774, -0.9458787787894589, 0.20914079384282933, 1.487644239755298, -1.4601155833251631, -1.653109915764168, -1.653166149925423, 1.9773117527941952, -0.25158374870191025, -0.05948675368658991, -0.7865757190025763, -0.9404477838814673, -2.0268601600545053], [0.1704062939055249, 0.2595598209214102, -1.9765340524052937, -1.043149176525692, -0.7441137561592083, -0.026002454439472283, 0.24412916699413742, 1.1257929683514525, 1.3644560191461277, 1.068454512203335, 1.341528531502859, -1.1664166902953155, -0.5544917649796093, 1.1436098002700195, 0.2886610787245648, 0.018721718675669165]], [[-1.9789881677034367, -1.8592429271083921, 0.7308894225311937, 0.34157639490844327, -0.29605762418597315, -0.41297173199761555, -1.7261026268818427, 0.3712151390391477, 1.034420799017549, 1.6421709791918642, 0.04913795013614401, -0.20419668212000805, -1.0959970457174089, 1.2232436651983958, 1.706855529223571, -0.6131700061123667], [0.2907587696302524, 0.0906701056389038, -0.5738273234080787, -2.4974495618193604, -0.3659991054773752, 0.6074441438068976, -1.4433020292385828, -0.8388812655122865, -3.2835604878882627, 1.2845277165004347, 0.5002763718002052, 0.04547852895218472, -0.2704733880742747, -0.637283176318838, -0.07880625474327216, -0.31031493210381433]], [[-0.24835672044910614, -0.03381111303746043, 0.1256509655342884, 1.54897841817735, -1.4494324054647152, 0.26196862488829153, -0.6245608174151475, -1.1394811867949461, -2.3681708051255725, 0.6068921428382499, -0.9903792389515337, -0.7596859730023938, 0.45987869267679876, -1.0934730110875337, 0.016045130164280344, -0.08488040408826328], [0.7918433900417065, 1.0006409423733347, -0.8768518458101385, 0.9071777346550384, -0.10674565894923128, 0.11279756399596276, -0.2225136131382745, 0.765539634412858, -1.2222004081820923, 0.250084266960452, 1.1657797884877044, -0.29366991481397603, 2.017772471985153, -0.7459058475973274, 0.2036961304237924, -0.49260852892014195]], [[-0.6287503105652976, -1.1257709935845257, -0.5500664691168721, -0.06223707311992144, -0.813097176129281, 0.8723957394787941, 1.5103266276144978, -1.5388961513985318, 0.8248433607793892, 0.1780600818362494, 0.23332070694060292, 2.4504993275728366, 1.1943246457946253, 0.3243098464303824, 0.17311415244634476, -2.1657171481499815], [0.7723909164870639, -0.7486869461197863, 0.1757326242531968, 0.0976306297693364, 0.9603515134967532, 2.0305028814674015, 0.9836022675228114, 1.2729486704011452, -0.2345432028281203, 0.4557735137751234, 0.7692489297752649, -1.4396281852276858, 2.0415429191664223, -0.714868246010375, -1.2775433055562428, 1.6340442221712168]], [[2.11843861535627, -0.7031057381505373, -0.6133099552415482, 3.1119129382202093, 0.11261711124014562, -1.9840581343909514, -0.6122687481297199, 2.0021187846260236, 0.659124064155549, 0.7520990806911315, 1.2245574277845088, 0.6757786896026271, -0.3332044808876784, -1.5943117200745838, 0.4463708390586508, -1.116034915943622], [-0.05623851864069769, 1.7589710766812527, 0.5406502875557727, 0.059491529994791854, 0.0656800558330705, 0.7307927491415821, 0.9212337664614076, -1.3168137745000827, 0.18547932372329412, 1.715647072880231, 0.7543686438915898, -0.7651259000530994, -0.6796276894565825, -2.061922492127639, -0.793378841071178, 1.6069652687673104]], [[-0.06952390178003978, -0.8550109597366835, -0.5761205222847363, -0.11074551402067748, -0.10654617191788289, -0.3539737132397008, -0.7690944405035621, 0.636570086422678, 1.538534566643514, -0.5679570038398728, -0.5412718676392838, 1.1893530639173242, -1.4637436480554302, -0.591394697898987, -0.37809709746053366, 0.16778159182906616], [-0.27493880956242583, -0.8866033394203041, 0.3105063468104034, -1.1877104022986131, -1.7425136392104728, -2.0282575532960294, 0.7486985378512533, -1.7368555460158381, 0.44705728629242797, 0.670709123171108, 0.8068993146395768, -0.17468272255542663, -0.5482001218387921, 0.0856713362219482, -0.5937349097040746, -1.286202262300972]], [[1.2554399352377776, -0.8944374586353495, 1.795889000036988, 0.5147645516042764, -0.4545567548775306, 0.4376476468228372, 1.2301494746406227, -1.378469250276322, -0.926772947442875, -1.0492675511760912, 0.16503139897918395, 0.4042508713136647, 0.028968342989254353, -0.7282098404051072, -0.5284513163621615, 1.1870432673580795], [1.116942674442043, 0.13135528984481942, -1.3106089423110556, 0.6281267915471365, 0.28765917679832087, 0.2527304950821622, 1.7300316088755747, 0.8205397313385244, -0.9744854542313894, -0.9647605552913328, 1.3803138486051605, 0.2057165176789978, 0.8870580070430005, -0.10639959608685703, -0.9494908518002053, 0.3242014124060831]], [[1.618915491436607, 0.3991358441971388, -1.0655647560808246, -0.24585874835996752, -0.4284127077441873, 0.24734402051603865, 1.0980541380077837, 2.054289901567956, 0.31449121970026706, -0.3364865964606304, 0.17159816274806688, 0.11086616939243021, -0.3297963407584658, 0.11860415855321144, 0.09529158095838645, -1.0805187404242564], [0.6438245074939835, -0.26670256295731154, -0.8877440710825989, 0.5859832203034813, -0.39159296460502474, 1.838255221034548, 0.5709067180842602, 0.1686367445319976, -0.6657762623743938, -0.31839120057526804, 0.9355282866027194, -0.6024860526591809, 0.8436412707458482, -2.0179640442538918, 0.2804525786513644, 0.8953021558456109]]]], "dtype": "float64"}, {"__nd__": [[2047, 2048]], "dtype": "int64"}, 4096, 10000.0, 4.0], "e": {"__tup__": [{"__nd__": [[[[0.06026131716287919, -0.34852658623536814, -0.1594945140041568, -0.026455367143477404, -0.16912882221834002, -0.5945022972956409, 0.24455348847557312, 1.4437882171118375, -0.872852369455626, 0.3376883250600867, -0.29333239336983435, -0.5693424162754767, -0.39290736446082786, 0.9349552603921054, 1.4604708308337964, -0.5761447107403301], [1.7927674952652173, 0.25198112826598507, 0.1446571549930497, -0.7570446322247762, -1.5560204510064892, -1.3449907514974044, 0.6632939769684848, -0.136456726483668, 0.21009515671235446, -1.172301128070158, -0.08478833558422515, -0.08224676304544512, 0.7793215116828522, 1.6700152011545273, 0.3273634590964987, 0.05333608007625058]], [[0.13731398436677272, -0.9915224999785692, 1.004178116249658, 0.3653188638661963, 0.5714241223944847, 0.5910684617094112, -0.10834649225029025, 0.661168433797484, 0.6884808096824453, 2.3226428953228337, -0.11346953670864568, 0.009354906343032682, 0.5597451218693191, 0.09796312502703575, -1.7929399422981243, 1.1844973473348628], [0.9864120955686688, 2.2118040545295305, 0.13528203905013814, 0.08475222601806798, -1.2217366787106627, 1.3664198286297542, 1.0887104778804877, -0.07890366856117542, -1.0286296547659277, -0.9439388079747167, -0.1674490336553629, 1.3422880524331102, 0.16961933664481593, 0.5284407381107334, -0.12191625903051617, -0.7462586910722313]], [[0.680580533402273, 0.5551494809601933, -1.6649171146286161, 0.611989841483057, 0.7265485211586241, 0.6837799475277638, 0.8016329622215574, 0.5663657068911601, -2.1045017082430424, -0.509709235744197, -2.013246285175686, 1.698020134959723, -1.8811682244565389, 1.071477758456069, -0.35969328175141285, -0.6312313260165782], [-0.04589618767367577, 1.2693530689547166, -0.07605890445147545, 0.533528876914636, -0.3126616603632286, 0.2901322819423102, 0.7927273061152563, 1.2577177383281029, -1.1368137462914676, 0.4806386311870449, 1.1392403055258882, 1.8384056678467746, 0.04958849593415415, 1.8756890343318826, -1.3311833738226606, -0.26459534812103785]], [[1.3148811927303288, -0.1151767388205018, 0.4732489813463479, 0.2157401865710773, 0.2094668576333904, -0.3123308666473026, -0.14720597334247304, -0.28960722700552904, -0.018467429758068088, 1.6624000856784216, -0.10103618837161915, -0.3241969865251747, 0.8382165341109455, -0.9665063728216139, -0.8432162305612829, -0.6719484371558411], [-0.4908328639733442, 0.19731122153493325, -0.03880155539172209, 1.5445384278747896, -0.25954997533185187, 0.5866078947274579, 0.04112009862755053, 1.3429606006772763, -0.684299738555379, 0.22149996367072236, -0.02498477984556937, 2.4207498057169827, 2.8289918636381772, 0.36589091162854764, 0.599507024614826, 0.42204960051261753]], [[-0.6411951839899818, 0.8589305631952887, -0.8763944898904493, 0.7116111815952577, -1.2694924443593762, -0.35511876684976296, 0.6746637100773902, -0.24646208437217076, -1.0337270260921847, -0.8610471628938425, 0.06126530256591556, -0.5370232047868593, 0.38683940478738876, -0.39445846307258314, 2.200736315980192, -0.7452441744129656], [-0.03021660314018089, 0.43482081130992434, -0.23999650062447073, 0.7890604222753745, 0.43310367404303507, -0.7049850023936611, -1.7944049706823062, 1.6170307597695315, -0.9711955126234657, 0.7447988737277245, 0.5714433212433045, 0.8176981776523471, -1.5663177112027564, -0.4113446848243048, 0.04417114362223151, 1.209094497015189]], [[-0.43533088926900476, 0.6792640262699349, 0.9177376893909369, -1.1576938807004855, 0.09419057948984749, -0.1914641755527595, 0.40896644935880494, 1.0263393075514207, -0.6159000300489972, 0.4922267968771014, -0.9575357530779001, -0.1750743368788522, -1.5263396435976007, -0.3507990369979407, -1.7028197289212559, 1.4561466305828843], [-0.2441588036865882, 0.8597944165436293, -0.43814531403376694, 0.594438984096371, 1.0133713281948804, -0.14833345073439153, -0.7176533288954967, 0.5555620296366314, 0.7136220054296636, -0.33710542720208686, -1.1485480751352388, -1.0768779434851576, -1.016188277870807, -1.5786961463343052, 1.9011078940498252, 0.9336652810034909]], [[-1.307404644143912, -1.1911168846746314, -0.6658794208086333, -0.9662994651946619, -0.23219384207159782, -0.0535889167114042, -0.21018222199688308, -0.881506224965462, 0.22123747010699707, 1.1456255428425066, -0.3011537571494098, 0.04976106432643168, -1.084940866156023, -0.05835031918082582, 1.9664582336924834, -1.1690685329089527], [0.5477317448770345, 0.6374059010094891, -0.45425843151167283, -2.2924600096772765, -0.7245275069865242, -1.4859898453771776, -0.09153647362894812, 1.636828673735408, -0.7011365634247376, -1.0003058900866544, -0.19102443349802076, 0.5622061875260608, -0.5552350173846015, 0.018923654890302094, 0.8913312263406776, 0.7801281462353002]], [[-0.49699992884614674, 0.6267253454616959, -0.15235600699897694, 0.9654648123454965, 1.3714557642520688, 1.5061304539075357, 0.5952783727499584, 0.34819917164882525, 0.5153932410160597, 0.5675130787056453, 0.5467891757706465, 0.09229742399296809, 0.16229314305734516, -0.6433475357016462, -1.4964891219020806, 0.6812461408874222], [-0.8665411493461783, 0.12373787561961518, 0.2175266443827817, -2.102687678362246, -0.2855227161764465, -0.266915240782824, 0.674066131152341, -1.0850687073751908, 0.9682897514327603, -2.01089882305099, 0.7795490555953624, 2.222548838456669, 0.8386000361795514, 0.7406008321160835, 0.1759683591910284, 0.09312746205803979]]]], "dtype": "float64"}, {"__nd__": [[[[0.10369181654050874, -1.6899671178738642, -1.3049220439019895, 0.1084292291276276, -0.4274623129333958, 0.7757582584377625, 1.7576012973835358, -1.1144618514434326, 1.7116865638296166, 0.9488840698901901, 1.5427722319612267, 0.33866324500737166, 0.8458731045789544, 0.24625263011638768, -0.09146090815421082, -2.2356373602984974], [-0.2783661715650241, 1.0913950339817151, -2.2534776212920287, 0.3822446668589905, -0.8040420162224885, -1.1410214965512366, 0.07140226381056528, 1.1080512001988931, -1.3465848683669905, -0.13350365195181702, -0.7926060001601865, 1.51742450044263, 0.4633387914738453, -0.08117541079961899, 0.3712492225855931, 0.19995711559616444]], [[1.5376534808704418, 1.5712390201521829, 0.4100785805867757, -0.3971482090782176, -1.123968029315653, -1.2022555167802516, -2.340823295423297, 0.4651613810506408, -1.6192721606071427, 1.91956185845983, 0.6070003656139749, 0.025377521026152428, -0.15986090871822917, -0.47058728016019064, 0.6429093060246452, -0.545346702040779], [-0.028734265252315483, 1.2863541872414779, -0.7433504850549215, 2.2329910648668743, -0.39340197564113955, 0.6072182992960529, -1.2196133272470882, -0.777886444545881, 3.2962834346788674, 0.059375310363046774, -0.1642689917174569, 1.1194076624054665, 0.22879266690229433, 0.6374983699307247, -0.7758057136887814, -0.44148591609003607]], [[1.001486645174568, 0.6051957259061278, 0.85878036217701, -1.7249290774228814, -0.1485959644850685, 1.0798010680764767, -0.5524049812764313, -1.1109166143427969, 2.1603098209234544, 0.05656321904373628, -0.5090535253046002, -0.03281151347998468, 1.513361143061399, 0.3135927929395064, -0.2918602362660105, -0.26736928401546667], [-0.6921483956754468, 0.3647914446315159, -1.4586823504240796, -0.9398095552282297, 1.8101415149433095, 0.7395914655622982, -0.29377505971912754, 0.8349370095393618, 1.2812963710543785, -0.9647547035690438, -0.012548916621206052, -0.16115663505550176, 0.8978797693924957, 0.148671745771423, 0.06856118339482903, -0.36275926552371657]], [[0.32793078762684397, 0.1356858300539239, -0.5219489009814634, 1.1760387425236019, 0.7771958873121052, -0.36536664135003305, 1.232062305317834, -1.16983987299519, -0.9839486375742555, 1.1316603192930288, -0.2908281580215505, -2.1507564445536285, 1.2179921716197595, 0.8560130957766047, 0.8905489922926774, -2.385373514175652], [-0.7512945340723015, 0.36558389931735946, -0.5142950489731437, -0.754955389803177, 2.2549801434345698, 0.6160203587228295, 1.4833676786335017, 0.9928888470818855, 0.2952198586438294, 0.7966241596317607, 0.5984366915464137, 1.2296761406883725, -0.07236762390646445, 2.0626433231269936, -0.6318311864992454, 1.8178752913298495]], [[-2.217742017874068, 0.7251825586585611, -1.3430799024387854, -2.4583568111549994, -0.26184971404592516, 1.6867194020071787, -0.7524197984976214, 2.1557795538839994, 0.06202452045261975, 0.7308357974361135, 0.2680044231083236, -2.0241444517205793, -0.2348245453136656, -1.9061201956599239, 0.0893554291974305, -0.7788633233201936], [0.04131139673988337, 1.9086208517448047, -0.28549018630013956, -0.4079735644090908, -0.5979094110416804, 2.0242427047272558, 1.1917942086854314, -1.5586379661056005, -0.18936398550714598, -1.547446468078507, 0.8831025636058858, 0.650011119763753, -0.3297150318049193, 0.8294720478100813, -0.24030855596916006, 1.3736752099210883]], [[-0.4321913391437506, -0.5996444267650236, 0.07336491398964512, 0.6424146692658166, -1.387258319870356, 0.607534846629433, -0.4854092901729895, 0.601219093655073, -1.4782194803359876, 0.8330939088303647, -0.7870881019877303, -1.007039558427236, -0.47896963152526795, -0.3254942219129049, -0.7062871190495563, 0.26815618379862804], [0.2385188636684238, 0.5630163249070679, -0.464494296220937, 0.9707854612648169, -1.1940487394588823, 0.012334694510636637, 0.9435738862269023, -1.5068013469240755, -0.46750435068037344, 0.9586077546443751, 0.7292089852391958, 0.7062192587890578, 1.382434361629964, -2.0300286046417617, -0.15079444556272903, -1.549365401763543]], [[-0.8878951350064181, -1.082095542038776, 0.972659144965966, -0.2729012712439301, -0.15252666407912072, 0.7066083826222858, 1.3313382434462728, -1.5517196901479202, 1.283230203951623, 0.8544296338850504, 1.5186793113810946, -0.5949169746272632, 0.42918134284382586, 0.47173046121733114, 0.14165805752195681, 0.9494288787185444], [-1.035916137789187, -0.9429297317103206, -1.8889760794193138, -0.4607781868925438, 0.9284148135763368, 0.09407599185783946, 1.973359669619971, 0.7575462612861508, 1.0602172390440718, -0.2426945864104927, -0.23394762597327556, -0.4738628645391968, 0.08755366233004308, 0.2575719024813943, 0.0198458982726476, 0.4522340859804488]], [[-1.6335614185374032, -0.3212704744655426, -0.789818203583563, 0.2693438588851136, -0.47194853648775836, -0.13021462954997326, 0.9107165228205756, 2.2015464979207024, 0.22642700269194893, -0.41148260468373016, -0.7355687487571319, 0.013846203153098466, 0.2637568073840817, 0.24143355434221878, 0.6208049536361365, -0.7354051710351215], [-0.5888442411828677, -0.3472551563872152, -1.2799604225864343, -0.7987409073726056, 0.619287951816557, 1.9268771024892635, 0.36029747863172257, 0.022106578019247952, 0.7148639644134125, 0.2278531765287682, -0.15812661821617124, 0.2614932905748562, 0.6939438567843046, 1.9335215989134635, 0.524188760388435, 0.9107774706680154]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[0.41629401487851075, 0.07274018953469119, 1.0863400275074944, -0.7792912035706152]], [[-0.9079190420445703, 0.18021408052490404, 1.2776237015139034, 0.5065013799515]], [[-0.11938237415906136, 0.3750716036436929, 0.5171626722510868, -1.1704579225994451]]], [[[-2.737101308424054, -0.9506103349906968, -1.4196819925965811, -0.5050193086315621]], [[-1.1275153636412767, -0.7912658410106157, 0.5087776489195652, 0.1735101418875656]], [[0.05337089932598016, 1.7401035051472364, -0.3785295647557631, -0.2968817458113318]]]], "dtype": "float64"}, {"__nd__": [[[[1.3511509205997787, -1.0253494858067438, -1.5425773917429322, 1.3973295584318948]], [[0.4284773896230237, 0.6939471341402738, 0.7324030743505717, 1.0005126796942065]], [[-0.373421250381592, 0.024766209961962146, -1.0105914896329433, 0.24289265325355808]]], [[[0.33745420956589356, -0.34638737757412497, 0.7765704945224762, 3.0646240121743187]], [[-1.0395714163213274, 0.9449557567607936, 1.060904820764859, 1.186570179751149]], [[-0.6814989053316384, 1.3214032858486306, 0.7814787108169615, 0.0047557062710890646]]]], "dtype": "float64"}, {"__nd__": [[0], [0]], "dtype": "int64"}, 1], "e": {"__tup__": [{"__nd__": [[[[0.41629401487851075, 0.07274018953469119, 1.0863400275074944, -0.7792912035706152]], [[-0.9079190420445703, 0.18021408052490404, 1.2776237015139034, 0.5065013799515]], [[-0.11938237415906136, 0.3750716036436929, 0.5171626722510868, -1.1704579225994451]]], [[[-2.737101308424054, -0.9506103349906968, -1.4196819925965811, -0.5050193086315621]], [[-1.1275153636412767, -0.7912658410106157, 0.5087776489195652, 0.1735101418875656]], [[0.05337089932598016, 1.7401035051472364, -0.3785295647557631, -0.2968817458113318]]]], "dtype": "float64"}, {"__nd__": [[[[1.3511509205997787, -1.0253494858067438, -1.5425773917429322, 1.3973295584318948]], [[0.4284773896230237, 0.6939471341402738, 0.7324030743505717, 1.0005126796942065]], [[-0.373421250381592, 0.024766209961962146, -1.0105914896329433, 0.24289265325355808]]], [[[0.33745420956589356, -0.34638737757412497, 0.7765704945224762, 3.0646240121743187]], [[-1.0395714163213274, 0.9449557567607936, 1.060904820764859, 1.186570179751149]], [[-0.6814989053316384, 1.3214032858486306, 0.7814787108169615, 0.0047557062710890646]]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[[[-0.21026098035303756, 0.38018947691790894, 1.0715250146520645, 1.0241744207011083, 2.4328604881410856, -0.20646864420110606, 0.7699827432898934, -0.2774447580275897], [0.9362072176514797, 0.28631535526570023, -0.44236699240730226, -1.1707319757858579, -0.6131279592631117, 0.39718727745368554, 1.6269680845754784, 0.9308051712316341]], [[-1.0413693363453491, -0.33361267495752045, 0.18406483453498276, 0.011326163811934926, 0.19017614050865247, 0.891305115149018, 0.3138678698529088, -0.6011586258767794], [-1.8446870347400581, -0.2656921423057748, -1.6127432706632423, -0.10446329880547667, 0.3634338444161454, -1.2998757087597277, -0.11584888412212659, -0.32716659712257967]]], [[[1.9060788306639675, 0.448768092469958, -1.9007844918971633, 0.4204667127804782, 0.38629809819080335, -0.23493279640908496, -0.41376384324268234, 0.5544675350795898], [0.6138501359810715, 0.3820300691786012, -1.0521268312007939, -0.3142268610437008, -1.6254592103256966, -1.5668897740611265, -0.6347933471206841, -1.1325814508998646]], [[-0.4956102868976653, 2.392772217170665, 0.4025426374596253, -0.980901437513614, -2.261459792935038, -0.9428343485501535, -1.5433629844530297, -2.311529977901388], [0.605044677013374, -0.602545854033424, 0.31645127035302306, 2.966238210900678, 0.14657582474666686, -0.00741881958223305, 0.22416597109845818, -0.39149740020283813]]], [[[-0.9520021536448358, -0.33749387102527995, 1.0182347009753103, -0.39336620261872396, -1.4920489584580787, -1.368557102107097, -1.1497435906932556, -0.1072094578765301], [0.365708100070376, 0.3683388201245052, 1.0810695463216409, -0.3066530564769316, 0.7394214562816036, 0.2368479351225408, 0.6244451117724273, -1.9157264678312913]], [[1.179128564389489, -1.6059455496125163, -1.1812087238869644, 0.7285397875893229, -0.4316533455947616, 1.7289521681788165, 0.1270528148073479, -1.0572758699119627], [0.12839329190039186, -0.4259935193114223, 1.5300532577009178, 1.6091804130260385, 0.7201277077487529, -0.42249883369204827, 0.06931524067415851, 1.0352593349259305]]], [[[-0.8136550558539355, -0.806729876970593, 0.8386394417799221, 0.7259921165979379, -0.24201310532798542, -0.27826632513265287, -1.9611249766590104, 1.1736487550830053], [-0.1261032036505945, -1.4362707237074057, -1.1350211878512138, -0.900301092920432, 1.6969619837176821, 0.6850574772130213, -1.3414423058366864, 1.85128431564108]], [[1.3351791816522625, -1.0612408977110352, -2.660992127169175, 0.9614986162699597, -0.6027781457301238, -0.13628479100503227, -0.5249081791935718, 1.259288122334379], [1.4789766959054405, 0.8101177912610477, -0.1506368553091435, 0.2618435842983696, 0.6211815737480053, -1.5746363883228522, 1.2433815123598415, -0.3186997372746957]]]], "dtype": "float64"}, {"__nd__": [[[[0.26307061762357126, 0.14000190648152444, 0.9083163692171554, 1.0714817689176728, 2.213199810675672, -0.3474785581572002, -0.8965520829041624, 0.1448122370416971], [-0.6418266810564457, 0.8751027737365411, -0.7909221451886773, 0.15036309924915556, 0.9588245333478886, 1.0343275199445572, 0.2264312628499982, 1.4341362820283172]]], [[[1.310883093403755, -1.6749932273685297, 0.38531661986118254, 0.20764310239580833, -0.26980638055382183, -0.60727273635687, 0.07507789217605107, 0.4187395477744383], [0.30865448854044664, 1.399886622511851, 1.028114000085676, -0.5532593446053599, -0.7224798425179273, 0.8842297105940803, -0.6599516844403207, 0.20378567909571624]]], [[[0.5136725133708302, 1.067847221996037, 0.554870562997444, 1.4836307120685868, -0.41991152579493224, 0.7100406584423806, -0.6179177338304227, 0.22685877595638926], [0.16443013069289658, -0.8879100723553962, -2.563278464806261, 0.9970925298778534, 0.4042292919261386, 0.18224069514569657, -0.6591445939728919, -0.2914599071297303]]], [[[0.4969172707031677, 0.06155714267584536, 1.2008442105159, 0.15820167857532091, -0.7803479255335084, 0.6506231924283851, -0.5935804333439826, 0.2554992282540186], [-1.7722288885231972, -0.312556799006476, 0.593013104847937, -0.6167822577566933, 0.0055620633214661895, 0.621496796021381, 0.09812398632467413, 1.8157239339574112]]]], "dtype": "float64"}, {"__nd__": [[144, 451], [289, 129], [192, 381], [261, 351]], "dtype": "int64"}, 500, 500000.0], "e": {"__tup__": [{"__nd__": [[[[1.0114187280465825, 0.08822880226390134, 0.8936596570939973, 1.0262691340060182, 2.2226227730259884, -0.42354340727773526, 0.9707789136935229, -0.26959321646134987], [-0.4341137650437131, 0.2875776732163457, -1.324156400636658, -1.1927189283003827, -1.031481986179325, -0.39627427105411195, 1.0437066306446718, 0.9024595193098672]], [[-0.8138055992380094, 0.4643985285451572, 0.11678395328680322, 0.015929667482213895, 0.6770062819117588, 0.8306962332093539, 0.3446076193195659, -0.6010542580721465], [0.024574694187926405, -1.151340351303566, -1.2266995771215214, -0.09658674437043938, 1.8799868350099522, 0.6593061262924967, -1.0533612716023855, -0.3295778570869258]]], [[[1.9156533983406117, -0.2900603432712871, -1.5797870976391817, 0.41189528843365014, 0.3356110647717301, -0.41527246105305426, -1.1351012856765172, 0.5608643110234572], [-0.91673486553199, -1.4989714437902653, -0.9195006075158023, -0.306449336799212, 1.4759833400803213, -0.5951261618474375, -0.8151393103386535, -1.1347105653421683]], [[-0.5554121799370602, -1.23927909993857, 0.9827564924025309, -0.945259050767027, -2.2475202472765696, -2.2535489354627423, -1.2562641256492952, -2.3263326924144114], [-0.5652541520628629, -0.09070600779819399, 0.2705308697601095, 2.968854290600524, -0.26086639714244075, 0.5957255788080413, 0.2778575851284171, -0.3711381674736468]]], [[[0.36043160662137674, 0.9030831898037883, 1.2892952271100497, -0.3922509847266099, 1.7328032929125499, -1.0822621711032951, -0.8345238450528213, -0.11122051160581525], [0.3272912694392694, -0.3023813195666457, 0.6074855211511063, -0.26777495765418724, -0.7572099638908109, 0.3167585332073281, 1.0906898748921834, -1.9215464003373164]], [[-1.2555654863846133, -2.344171102339695, -1.1720076900598664, 0.7392975982983387, -0.014969686774874091, -0.27055230755987836, -0.19440792570694587, -1.04978166699038], [0.46609636190873616, 0.4955955929724267, 1.2777039329289257, 1.5878743563211335, -0.5637578684931439, -0.3381726647628116, 0.8445947156884558, 1.0676509358192303]]], [[[0.7294382323677535, 0.6401346495817378, 1.4897011731955065, 0.7096315728508185, 0.43419437839218133, 0.5643340079285362, -1.5264723063209087, 1.1836126834752638], [1.2018021953353224, -1.5631563226440204, -0.35916677484030923, -0.9347005577236412, 1.204679822791597, -0.2978584392668566, -1.720098771760051, 1.8341566298011467]], [[-1.4422422447106744, 0.9295275990236802, -2.292392259736103, 0.9439266840761394, 0.25687009880359213, 0.5298907716568402, -1.4496018159972917, 1.2725127029741263], [1.4368569479016915, 1.5849378770229139, -0.7246209166309203, 0.2677468561063177, -0.7132185680855891, -0.7897737126077398, 1.0215740671320113, -0.31375660347211864]]]], "dtype": "float64"}, {"__nd__": [[[[1.3159021834845064, -0.17462092182734534, 1.070867066738769, 1.070341336337621, 1.7988499090620054, -0.3314353871535694, -0.6943255643586322, 0.15301368843002702], [0.8272587392943728, 0.708345223448564, -0.770253744861767, 0.11592465060346176, 0.8043189370683833, -1.1549395341461173, -0.28902962663389387, 1.4373299594833058]]], [[[1.303266457491143, -0.38947930020633925, 0.3237423401483672, 0.2011828556246523, -0.3044774348582144, 1.7385880371047029, 0.22202811675574938, 0.42188141159051734], [-0.4426032491089164, 1.0693929174049819, 1.1307831486008766, -0.5546444057012776, 0.6491125326313799, -1.2640979096529628, -0.4624760467398002, 0.19998522100878138]]], [[[-0.62926985229803, 0.06002639283642763, 0.7002696414351758, 1.4812369220052948, 0.21024888460334146, 1.2809575552409962, -0.4464595129991922, 0.24199622912320934], [0.2018870962685052, -0.010609016942546512, -1.8618858604006183, 1.0027932077837534, -0.38688523969138666, -0.9063572233489754, -1.8810234253574514, -0.2711976641216867]]], [[[-0.673142111313088, 0.1906722641583104, 1.3341211213105635, 0.15464003043214194, 0.6346253672715058, -0.6250951191785625, -0.12035523441641416, 0.25767050225121707], [-1.1542372637892915, -0.6197668617336212, 0.47470931793825144, -0.6505673444616507, 1.3448280591751376, 0.31597319058248463, 0.36870573995563893, 1.8038946444150603]]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(rope_decode, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")